# Coding Practice: Choosing and Checking Visualizations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module03/week3_visualization_practice.ipynb)

**Student version · Estimated time: 35–45 minutes**

A plotting library can draw a chart quickly. Your job as an analyst is to decide whether the chart fits the question, represents the data honestly, and communicates evidence clearly.

In this practice, you will use Palmer Penguins to compare visualization choices, interpret patterns, repair a simulated AI-generated draft, and complete one short independent analysis. **No live AI service is used or required.**

## Learning objectives

By the end of this practice, you will be able to:

1. Match an analytical question to a univariate, bivariate, or grouped visualization.
2. Compare distribution and aggregated views without treating either as universally superior.
3. Interpret visible patterns without making claims the chart cannot support.
4. Recognize weak scale, aggregation, labeling, and accessibility choices.
5. Check a visual impression against numeric evidence.
6. Validate a simulated AI-generated chart and claim.

### Suggested pacing

- Set up and inspect the data: 5 minutes
- Compare univariate and grouped views: 10–12 minutes
- Examine a relationship: 7–8 minutes
- Critique and repair the simulated AI draft: 8–10 minutes
- Complete the mini-analysis and checklist: 8–10 minutes

## 1. Set up and inspect the data

The notebook uses the course copy of the Palmer Penguins dataset. When you run it inside the repository, pandas reads the relative path `../data/penguins.csv`. When you open the notebook directly in Colab, the same code uses the course repository's raw GitHub URL.

**Dataset citation:** Horst AM, Hill AP, Gorman KB (2020). *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. R package version 0.1.1. <https://allisonhorst.github.io/palmerpenguins/>

The CSV stores missing measurements as `NA`. We explicitly ask pandas to interpret that token as a missing value. We do not change or delete any source rows.

| Column | Meaning |
| --- | --- |
| `species` | Adelie, Chinstrap, or Gentoo |
| `island` | Island where the observation was recorded |
| `bill_length_mm` | Bill length in millimeters |
| `bill_depth_mm` | Bill depth in millimeters |
| `flipper_length_mm` | Flipper length in millimeters |
| `body_mass_g` | Body mass in grams |
| `sex` | Recorded sex when available |
| `year` | Observation year |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

LOCAL_DATA_PATH = Path("../data/penguins.csv")
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "obscrivn/DataScience-book/main/data/penguins.csv"
)

data_source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.exists() else DATA_URL
penguins = pd.read_csv(data_source, na_values=["NA"])

sns.set_theme(style="whitegrid", context="notebook")

print(f"Loaded {penguins.shape[0]} rows and {penguins.shape[1]} columns.")
print(f"Data source: {data_source}")
penguins.head()

In [ ]:
EXPECTED_COLUMNS = [
    "species",
    "island",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "sex",
    "year",
]

assert penguins.shape == (344, 8), "The course dataset should contain 344 rows and 8 columns."
assert penguins.columns.tolist() == EXPECTED_COLUMNS, "Unexpected columns or column order."
assert set(penguins["species"].unique()) == {"Adelie", "Chinstrap", "Gentoo"}

data_overview = pd.DataFrame(
    {
        "data_type": penguins.dtypes.astype(str),
        "missing_values": penguins.isna().sum(),
        "unique_values": penguins.nunique(dropna=True),
    }
)
data_overview

In [ ]:
species_counts = (
    penguins["species"]
    .value_counts()
    .rename_axis("species")
    .to_frame("observations")
)
species_counts

### Think before plotting

Classify each question. Is it primarily univariate, bivariate, or grouped/multivariate? Which variables would you need?

1. What is the shape of the body-mass distribution?
2. How does body mass vary across species?
3. Is flipper length associated with body mass, and does the pattern differ by species?

> **Write a short classification and chart choice for each question:**
>
> 1.
> 2.
> 3.

## 2. One variable, two useful views

A histogram and a boxplot can examine the same numeric variable, but they summarize it differently.

**Predict:** Before running the next cell, where do you expect most body-mass observations to fall? Do you expect one smooth peak or more than one concentration?

In [ ]:
mass_data = penguins.dropna(subset=["body_mass_g"]).copy()

print(
    f"Body-mass plot uses {len(mass_data)} of {len(penguins)} rows "
    "(rows missing body_mass_g are excluded only from these plots)."
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(
    data=mass_data,
    x="body_mass_g",
    bins=18,
    color="#0072B2",
    edgecolor="white",
    ax=axes[0],
)
axes[0].set(
    title="Distribution of penguin body mass",
    xlabel="Body mass (g)",
    ylabel="Number of penguins",
)

sns.boxplot(
    data=mass_data,
    x="body_mass_g",
    color="#56B4E9",
    ax=axes[1],
)
axes[1].set(
    title="Compact summary of body mass",
    xlabel="Body mass (g)",
    ylabel="",
)

fig.tight_layout()
plt.show()

### Worked interpretation

The histogram reveals multiple concentrations in the combined body-mass distribution. The boxplot more compactly shows the median, middle half, range, and potential extreme values, but it hides the detailed shape.

Neither plot explains *why* the distribution has this structure. A useful next step is to examine a relevant grouping variable. Also remember the Week 02 principle: an unusual point is a reason to investigate, not an automatic deletion.

> **Your observation:** What did you notice that your prediction missed?
>

## 3. Grouped comparison: distribution and aggregation

The question now changes from “What is the overall distribution?” to “How does body mass vary across species?”

The two charts below answer related but different questions:

- A **distribution view** shows within-species spread, overlap, median, and possible unusual values.
- An **aggregated view** makes the mean easy to compare, but compresses each species to one number.

Neither view is universally superior. Choose based on the analytical question, and explain what the chosen summary leaves out.

In [ ]:
SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = ["#0072B2", "#D55E00", "#009E73"]

grouped_data = penguins.dropna(subset=["species", "body_mass_g"]).copy()
group_summary = (
    grouped_data.groupby("species", observed=True)["body_mass_g"]
    .agg(observations="size", mean_g="mean", median_g="median", std_g="std")
    .reindex(SPECIES_ORDER)
    .round(1)
)
display(group_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(
    data=grouped_data,
    x="species",
    y="body_mass_g",
    order=SPECIES_ORDER,
    color="#56B4E9",
    ax=axes[0],
)
axes[0].set(
    title="Body-mass distributions by species",
    xlabel="Species",
    ylabel="Body mass (g)",
)

axes[1].bar(
    group_summary.index,
    group_summary["mean_g"],
    color=SPECIES_COLORS,
)
axes[1].set(
    title="Mean body mass by species",
    xlabel="Species",
    ylabel="Mean body mass (g)",
)
axes[1].set_ylim(0, group_summary["mean_g"].max() * 1.15)

for index, value in enumerate(group_summary["mean_g"]):
    axes[1].text(index, value + 75, f"{value:,.0f}", ha="center")

fig.tight_layout()
plt.show()

### Compare, then modify

1. Which chart better supports a question about individual variation and overlap?
2. Which chart makes the group means easiest to compare?
3. What information is hidden by each chart?
4. Do the unequal species counts affect how confidently you read the comparison?

The next cell is a **guided modification**. Change `guided_measure` to `"bill_depth_mm"` or `"flipper_length_mm"`, update the y-axis label, and rerun it.

In [ ]:
guided_measure = "bill_length_mm"  # Try "bill_depth_mm" or "flipper_length_mm".
guided_label = "Bill length (mm)"  # Update this label when you change the measure.

guided_data = penguins.dropna(subset=["species", guided_measure]).copy()

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(
    data=guided_data,
    x="species",
    y=guided_measure,
    order=SPECIES_ORDER,
    color="#E69F00",
    ax=ax,
)
ax.set(
    title=f"{guided_label} by species",
    xlabel="Species",
    ylabel=guided_label,
)
fig.tight_layout()
plt.show()

print(f"This comparison uses {len(guided_data)} of {len(penguins)} rows.")

> **Guided interpretation (2–3 sentences):**
>
> Which groups overlap? Which differ most? What does the distribution view reveal that a table of group means would not?
>

## 4. Two numeric variables with a grouping variable

A scatterplot fits a question about association between two numeric variables. Species adds a third variable.

This plot uses both color and marker shape for species. That redundant encoding is a lightweight accessibility improvement: readers do not have to rely on color alone. Smaller, partly transparent markers also reduce overplotting.

In [ ]:
species_palette = {
    "Adelie": "#0072B2",
    "Chinstrap": "#D55E00",
    "Gentoo": "#009E73",
}
species_markers = {
    "Adelie": "o",
    "Chinstrap": "s",
    "Gentoo": "^",
}

relationship_data = penguins.dropna(
    subset=["flipper_length_mm", "body_mass_g", "species"]
).copy()

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(
    data=relationship_data,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",
    style="species",
    hue_order=SPECIES_ORDER,
    style_order=SPECIES_ORDER,
    palette=species_palette,
    markers=species_markers,
    s=60,
    alpha=0.72,
    ax=ax,
)
ax.set(
    title="Flipper length and body mass by species",
    xlabel="Flipper length (mm)",
    ylabel="Body mass (g)",
)
ax.legend(title="Species", frameon=True)
fig.tight_layout()
plt.show()

overall_correlation = relationship_data[
    ["flipper_length_mm", "body_mass_g"]
].corr().iloc[0, 1]
print(f"Overall Pearson correlation: {overall_correlation:.3f}")
print(f"Plot uses {len(relationship_data)} of {len(penguins)} rows.")

### Interpret without overclaiming

> **Write 2–3 sentences:**
>
> - What association is visible?
> - How does species help explain the clusters?
> - What causal claim would be inappropriate?
> - What might you investigate next?
>
> Remember: a correlation and a scatterplot describe association in these observations. They do not show that changing flipper length would cause body mass to change.

## 5. Critique a simulated AI-generated draft

Imagine an AI coding assistant returned the chart below with this claim:

> “Species determines body mass, and every Gentoo penguin is heavier than every Adelie or Chinstrap penguin.”

This is a **simulated draft**. No live AI connection is used. Run the code, then audit the data, code, chart, and claim before trusting it.

In [ ]:
ai_summary = (
    grouped_data.groupby("species", observed=True)["body_mass_g"]
    .mean()
    .reindex(SPECIES_ORDER)
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ai_summary.index, ai_summary.values, color=SPECIES_COLORS)
ax.set_ylim(ai_summary.min() - 200, ai_summary.max() + 200)
ax.set(
    title="Species determines penguin body mass",
    xlabel="Species",
    ylabel="Mean body mass (g)",
)
fig.tight_layout()
plt.show()

### Audit before repair

> **Write one concise note for each check:**
>
> 1. **Data:** Does a mean describe every individual?
> 2. **Code:** What transformation or aggregation did the code apply?
> 3. **Chart:** How does the truncated y-axis affect the visual comparison?
> 4. **Claim:** Which words imply causation or certainty that the evidence does not support?
> 5. **Accessibility:** Can the comparison still be read without distinguishing the bar colors?
>

In [ ]:
claim_check = (
    grouped_data.groupby("species", observed=True)["body_mass_g"]
    .agg(
        observations="size",
        minimum_g="min",
        maximum_g="max",
        mean_g="mean",
        median_g="median",
    )
    .reindex(SPECIES_ORDER)
    .round(1)
)
claim_check

### Compare two defensible repairs

The overlap in the minimum and maximum values disproves the word “every.” The next cell shows two possible repairs:

- The boxplot supports a question about distributions and overlap.
- The point-and-error-bar view supports a question about means while retaining a measure of within-group variation.

These are alternatives for different questions—not a ranking in which one chart type always wins.

In [ ]:
repair_summary = (
    grouped_data.groupby("species", observed=True)["body_mass_g"]
    .agg(mean_g="mean", std_g="std")
    .reindex(SPECIES_ORDER)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(
    data=grouped_data,
    x="species",
    y="body_mass_g",
    order=SPECIES_ORDER,
    color="#56B4E9",
    ax=axes[0],
)
axes[0].set(
    title="Body-mass distributions overlap across species",
    xlabel="Species",
    ylabel="Body mass (g)",
)

x_positions = range(len(repair_summary))
axes[1].errorbar(
    x_positions,
    repair_summary["mean_g"],
    yerr=repair_summary["std_g"],
    fmt="o",
    color="#0072B2",
    ecolor="#4D4D4D",
    capsize=5,
    markersize=7,
)
axes[1].set_xticks(list(x_positions), repair_summary.index)
axes[1].set(
    title="Species means with ±1 standard deviation",
    xlabel="Species",
    ylabel="Body mass (g)",
)

fig.tight_layout()
plt.show()

> **Repair reflection (2–3 sentences):**
>
> Choose one repaired view. What question does it answer well? What information or uncertainty still remains?
>

## 6. Short independent mini-analysis

Spend **no more than 8–10 minutes** on this section. Choose **one** question:

- How does bill depth vary across islands?
- How are bill length and bill depth related, and how does species affect the pattern?

Before coding, write one sentence naming your variables and why your chart type fits the question. Then create **one chart** in the next cell. Use the patterns above as a reference.

In [ ]:
# Your short independent analysis goes here.
# Keep this cell self-contained: create one plot and call plt.show().


### Independent interpretation

> **Write three short statements:**
>
> 1. **Evidence:** What visible pattern supports your interpretation?
> 2. **Limit:** What conclusion does the chart not support?
> 3. **Check:** What numeric summary or alternative view would you use to validate the pattern?
>

## 7. Reusable chart check

Before sharing any exploratory chart, ask:

- **Question:** Does the chart type match the variables and analytical question?
- **Data:** Are filtering, missing values, grouping, and aggregation visible and justified?
- **Scale:** Are axes and comparisons honest?
- **Access:** Are labels and units readable, and can categories be distinguished without color alone?
- **Evidence:** Does the written claim match what the chart actually shows?
- **Limit:** Have I stated what the visualization cannot establish?

This lightweight check applies whether code was written by you, adapted from documentation, or suggested by AI.

In [ ]:
# Final source-data check: the provided workflow did not alter or delete source rows.
assert penguins.shape == (344, 8)
assert penguins.isna().sum()["sex"] == 11
assert penguins.isna().sum()["body_mass_g"] == 2

print("Validation check passed: the source DataFrame still has 344 rows and expected missing values.")

## Takeaways

- Start with the analytical question and variable types, not a favorite chart.
- Distribution and aggregated views answer different questions.
- Grouping can reveal structure hidden in an overall distribution or correlation.
- Accessible labels and redundant encodings improve interpretation with little extra code.
- A polished chart can still mislead through aggregation, scale, or an overconfident claim.
- Validate visual impressions with the underlying data before communicating a conclusion.